# Proposal Indicator Analysis

This notebook parses a folder of PDF and Word proposal documents, identifies candidate indicator sections, extracts structured M&E indicators with an LLM, matches them against a reerence library, and produces validation outputs and charts.


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import matplotlib.pyplot as plt
import extraction_utils

ROOT = Path('.')
PROPOSALS_DIR = ROOT / 'proposals'
OUTPUTS_DIR = ROOT / 'outputs'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
OUTPUTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

# Reference library: built from the ZOA IndiKit export (see "Build the reference library" cell).
ZOA_INDICATORS_PATH = ROOT / 'MEAL_Proposal-main' / 'MEAL_Proposal-main' / 'notebooks' / 'data' / 'zoa_indikit_net' / 'indicators.csv'
REFERENCE_LIBRARY_PATH = ROOT / 'reference_library.csv'
ENV_PATH = ROOT / '.env'
load_dotenv(dotenv_path=ENV_PATH)   # loads OPENAI_API_KEY / AZURE_OPENAI_* from .env

# ---- LLM provider / model -------------------------------------------------------------
# Exactly one option should be active. Uncomment the block you want and comment out the others.

# Option A (active): OpenAI gpt-4o-mini, as specified in the plan. Requires OPENAI_API_KEY in .env.
# All cached extraction responses in outputs/api_cache were produced with this model, so re-runs are free.
LLM_PROVIDER = 'openai'
MODEL_NAME = 'gpt-4o-mini'

# Option B: local Llama 3.2 through Ollama. Install Ollama (https://ollama.com), then: ollama pull llama3.2
# LLM_PROVIDER = 'ollama'
# MODEL_NAME = 'llama3.2'            # e.g. 'llama3.2' (3B) or 'llama3.2:1b'
# OLLAMA_BASE_URL = 'http://localhost:11434/v1'

# Option C: Azure OpenAI. Requires AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY in .env
#           (optionally AZURE_OPENAI_API_VERSION). MODEL_NAME is your *deployment name*, not the base model.
# LLM_PROVIDER = 'azure'
# MODEL_NAME = 'gpt-4o-mini'       # <- replace with the deployment name from the Azure portal
# AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT', 'https://<your-resource>.openai.azure.com/')
# AZURE_OPENAI_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2024-10-21')
# ----------------------------------------------------------------------------------------

FORCE_REFRESH = False           # re-call the LLM even when a cached response exists
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
MIN_CHUNK_SCORE = 0.30
KEEP_NEIGHBORS = 1
LINK_TOP_K = 5                  # number of ZOA candidates shown to the LLM per proposal indicator (Phase 4)

extraction_utils.FORCE_REFRESH = FORCE_REFRESH  # the cache helpers read the module-level flag
llm_config = extraction_utils.configure_llm(
    provider=LLM_PROVIDER,
    model_name=MODEL_NAME,
    base_url=globals().get('OLLAMA_BASE_URL'),
    azure_endpoint=globals().get('AZURE_OPENAI_ENDPOINT'),
    azure_api_version=globals().get('AZURE_OPENAI_API_VERSION'),
)

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')   # kept for reference; provider readiness is checked below

LLM_AVAILABLE, llm_status = extraction_utils.llm_available()
print(f'LLM provider: {llm_config["provider"]} | model: {llm_config["model"]} | endpoint: {llm_config["base_url"]}')
print(f'LLM ready: {LLM_AVAILABLE} ({llm_status})')

proposal_paths = sorted(PROPOSALS_DIR.rglob('*.pdf')) + sorted(PROPOSALS_DIR.rglob('*.docx'))
proposal_paths = [p for p in proposal_paths if p.is_file()]
print(f'Found {len(proposal_paths)} proposal documents')


LLM provider: openai | model: gpt-4o-mini | endpoint: https://api.openai.com/v1
LLM ready: True (OpenAI configured, model gpt-4o-mini)
Found 108 proposal documents


In [3]:
from extraction_utils import extract_chunks_from_file, filter_indicator_chunks, LLMClient, deduplicate_indicators, match_indicators_to_library

all_chunks = []
parse_errors = []
for path in proposal_paths:
    chunks, errors = extract_chunks_from_file(str(path))
    all_chunks.extend(chunks)
    parse_errors.extend(errors)

print(f'Parsed {len(all_chunks)} chunks from {len(proposal_paths)} documents')
if parse_errors:
    print(f'Encountered {len(parse_errors)} parse issues')


Parsed 10385 chunks from 108 documents


## Section targeting and candidate selection
Score all chunks for indicator likelihood, keep the highest-scoring chunks plus neighbors, and record coverage for each document.


In [4]:
chunks_by_doc = {}
coverage_records = []
candidate_chunks = []
for path in proposal_paths:
    doc_id = path.name
    doc_chunks = [chunk for chunk in all_chunks if chunk.doc_id == doc_id]
    selected, summary = filter_indicator_chunks(doc_chunks, min_score=MIN_CHUNK_SCORE, keep_neighbors=KEEP_NEIGHBORS)
    chunks_by_doc[doc_id] = {
        'all_chunks': doc_chunks,
        'selected_chunks': selected,
        'summary': summary,
    }
    coverage_records.append({
        'doc_id': doc_id,
        'total_chunks': summary['total_chunks'],
        'selected_chunks': summary['selected_chunks'],
        'coverage_fraction': summary['coverage_fraction'],
    })
    candidate_chunks.extend(selected)
coverage_df = pd.DataFrame(coverage_records)
print(f'Selected {len(candidate_chunks)} candidate chunks across {len(chunks_by_doc)} documents')
coverage_df.head()


Selected 12475 candidate chunks across 103 documents


,doc_id,total_chunks,selected_chunks,coverage_fraction
0,2023_00217_MR_01_01_18-Jan-2024.pdf,29,29,1.0
1,Syria Joint Response 2024-2026 ZOA - Annex II ...,1,1,1.0
2,Syria Joint Response 2024-2026 ZOA - Annex II ...,3,3,1.0
3,CBPF-SYR-24-S-INGO-33359 - Revised proposal 31...,37,37,1.0
4,CBPF-SYR-24-S-INGO-33280 Revision submitted.pdf,20,20,1.0


## LLM extraction
Extract structured indicators from the candidate chunks with the model chosen in the config cell. Responses are cached per chunk **and per model** under `outputs/api_cache`, so switching models never reuses another model's answers, and re-running with the same model is free.

If the LLM server is not reachable, this cell falls back to **cache-only mode**: it rebuilds the dataset from the cached responses and reports how many chunks had no cached answer. The save cells further down never overwrite an existing non-empty output file with an empty one.


In [5]:
extracted_indicators = []
llm_failures = []
cache_misses = 0
client = LLMClient(model_name=MODEL_NAME)
if not LLM_AVAILABLE:
    print(f'LLM not available ({llm_status}).')
    print(f'Rebuilding from cached {MODEL_NAME} responses only; chunks without a cached response are counted as misses and skipped.')
for n, chunk in enumerate(candidate_chunks, 1):
    try:
        chunk_indicators = client.extract_indicators_from_chunk(chunk, prompt_version='v1', cache_only=not LLM_AVAILABLE)
        extracted_indicators.extend(chunk_indicators)
    except extraction_utils.CacheMiss:
        cache_misses += 1
    except Exception as exc:
        llm_failures.append({'doc_id': chunk.doc_id, 'source_file': chunk.source_file, 'error': str(exc)})
    if n % 500 == 0:
        print(f'  {n} / {len(candidate_chunks)} chunks processed, {len(extracted_indicators)} indicators so far')
print(f'Extracted {len(extracted_indicators)} indicators from {len(candidate_chunks)} chunks with {MODEL_NAME}')
if cache_misses:
    print(f'WARNING: {cache_misses} chunks had no cached response and were skipped. Start the LLM and re-run this cell to extract them.')
if llm_failures:
    print(f'Encountered {len(llm_failures)} extraction failures. First error: {llm_failures[0]["error"][:200]}')


  500 / 12475 chunks processed, 144 indicators so far
  1000 / 12475 chunks processed, 208 indicators so far
  1500 / 12475 chunks processed, 272 indicators so far
  2000 / 12475 chunks processed, 484 indicators so far
  2500 / 12475 chunks processed, 548 indicators so far
  3000 / 12475 chunks processed, 612 indicators so far
  3500 / 12475 chunks processed, 676 indicators so far
  4000 / 12475 chunks processed, 740 indicators so far
  4500 / 12475 chunks processed, 804 indicators so far
  5000 / 12475 chunks processed, 887 indicators so far
  5500 / 12475 chunks processed, 951 indicators so far
  6000 / 12475 chunks processed, 1015 indicators so far
  6500 / 12475 chunks processed, 1147 indicators so far
  7000 / 12475 chunks processed, 1211 indicators so far
  7500 / 12475 chunks processed, 1275 indicators so far
  8000 / 12475 chunks processed, 1339 indicators so far
  8500 / 12475 chunks processed, 1487 indicators so far
  9000 / 12475 chunks processed, 1551 indicators so far
  95

## Deduplication and assembly
Deduplicate extracted indicators within each proposal and assemble the final indicator dataset.


In [6]:
deduped_indicators = []
for doc_id in sorted({ind.doc_id for ind in extracted_indicators}):
    items = [ind for ind in extracted_indicators if ind.doc_id == doc_id]
    keep, merged = deduplicate_indicators(items)
    deduped_indicators.extend(keep)
indicators_df = pd.DataFrame([ind.model_dump() for ind in deduped_indicators])
if indicators_df.empty:
    indicators_df = pd.DataFrame(columns=['indicator_text','level','baseline','target','means_of_verification','sector','disaggregation','unit_or_type','source_confidence','doc_id','chunk_ref'])
print('Assembled indicators dataframe with', len(indicators_df), 'rows')
indicators_df.head()


Assembled indicators dataframe with 988 rows


,indicator_text,level,baseline,target,means_of_verification,sector,disaggregation,unit_or_type,source_confidence,doc_id,chunk_ref
0,Direct project participants CBS Kalehe,unclear,NaN,72,NaN,NaN,NaN,NaN,low,(ENGLISH version) 2024 Project proposal SelfRi...,proposals/africa/DRC/small/selfrise/(ENGLISH v...
1,Direct project participants VSLA Kalehe,unclear,NaN,200,NaN,NaN,NaN,NaN,low,(ENGLISH version) 2024 Project proposal SelfRi...,proposals/africa/DRC/small/selfrise/(ENGLISH v...
2,Direct project participants PIP Idjwi,unclear,NaN,2100,NaN,NaN,NaN,NaN,low,(ENGLISH version) 2024 Project proposal SelfRi...,proposals/africa/DRC/small/selfrise/(ENGLISH v...
3,Direct project participants G/PM Idjwi,unclear,NaN,20,NaN,NaN,NaN,gender champions,low,(ENGLISH version) 2024 Project proposal SelfRi...,proposals/africa/DRC/small/selfrise/(ENGLISH v...
4,Direct project participants VLSA Idjwi,unclear,NaN,1100,NaN,NaN,100 1e generation and 1000 2e and 3e generation,NaN,low,(ENGLISH version) 2024 Project proposal SelfRi...,proposals/africa/DRC/small/selfrise/(ENGLISH v...


## Validation and outputs
Run basic QA checks on the extracted dataset and persist results to CSV and Excel.


In [7]:
qa_checks = []
qa_checks.append({'check': 'missing_indicator_text', 'count': int(indicators_df['indicator_text'].isna().sum() if not indicators_df.empty else 0)})
qa_checks.append({'check': 'missing_level', 'count': int(indicators_df['level'].isna().sum() if not indicators_df.empty else 0)})
qa_checks.append({'check': 'duplicate_rows', 'count': int(indicators_df.duplicated(subset=['indicator_text', 'doc_id']).sum() if not indicators_df.empty else 0)})
qa_checks.append({'check': 'proposals_with_zero_indicators', 'count': int((coverage_df['selected_chunks'] == 0).sum())})
qa_df = pd.DataFrame(qa_checks)
qa_df


,check,count
0,missing_indicator_text,0
1,missing_level,0
2,duplicate_rows,0
3,proposals_with_zero_indicators,0


In [8]:
indicators_csv = OUTPUTS_DIR / 'indicators_extracted.csv'
indicators_xlsx = OUTPUTS_DIR / 'indicators_extracted.xlsx'
if indicators_df.empty and indicators_csv.exists() and len(pd.read_csv(indicators_csv)) > 0:
    print(f'No indicators extracted in this run; keeping the existing {indicators_csv} untouched.')
else:
    indicators_df.to_csv(indicators_csv, index=False)
    with pd.ExcelWriter(indicators_xlsx, engine='openpyxl') as writer:
        for doc_id, group in indicators_df.groupby('doc_id'):
            safe_name = str(doc_id)[:31].replace('/', '_').replace('\\', '_')
            group.to_excel(writer, sheet_name=safe_name or 'doc', index=False)
        indicators_df.to_excel(writer, sheet_name='Combined', index=False)
    print('Saved', indicators_csv, 'and', indicators_xlsx)


Saved outputs/indicators_extracted.csv and outputs/indicators_extracted.xlsx


## Matching against the reference library
Compute lexical and embedding similarity between extracted indicators and the ZOA reference indicator library.

The library is built from the ZOA IndiKit export (`indicators.csv`, 303 indicators across 30 sectors). Each IndiKit row stores its English and French wording in one text block; the cell below splits them, uses the English wording as the match text (falling back to the indicator title when wording is missing), and writes a standardised `reference_library.csv` with the columns the matcher expects (`indicator_id`, `indicator_text`, `sector`, `level`). IndiKit has no result-level attribute, so `level` is left blank. The French wording is kept in `indicator_text_fr` for a later bilingual matching step.

If the IndiKit file is missing, the matcher falls back to a small dummy library and marks this clearly.


In [9]:
# Build the reference library from the ZOA IndiKit export
from extraction_utils import build_reference_library_from_zoa

if ZOA_INDICATORS_PATH.exists():
    reference_df = build_reference_library_from_zoa(str(ZOA_INDICATORS_PATH), str(REFERENCE_LIBRARY_PATH))
    print(f'Built {REFERENCE_LIBRARY_PATH} with {len(reference_df)} ZOA indicators across {reference_df["sector"].nunique()} sectors')
    print(f'  match text from wording: {(reference_df["text_source"] == "wording").sum()}, from title: {(reference_df["text_source"] == "title").sum()}')
    print(f'  with French wording: {(reference_df["indicator_text_fr"] != "").sum()}')
    display(reference_df[['indicator_id', 'indicator_text', 'sector']].head())
else:
    print(f'ZOA indicator file not found at {ZOA_INDICATORS_PATH}. The matcher will fall back to a dummy library.')


Built reference_library.csv with 303 ZOA indicators across 30 sectors
  match text from wording: 294, from title: 9
  with French wording: 294


,indicator_id,indicator_text,sector
0,5083-individual-dietary-diversity-score,the average number of different food groups co...,Food Security
1,5089-household-dietary-diversity-score,the average number of different food groups co...,Food Security
2,5090-fsl-food-consumption-score,% of the target population with acceptable Foo...,Sector specific results of CVA
3,5091-fsl-coping-strategy-index,% of households in the highest Coping Strategy...,Sector specific results of CVA
4,5092-months-of-adequate-household-food-provisi...,the average duration of a period in the past 1...,Food Security


In [10]:
matches_df, using_dummy_library = match_indicators_to_library(deduped_indicators, str(REFERENCE_LIBRARY_PATH), embedding_model=EMBEDDING_MODEL)
if using_dummy_library:
    print('WARNING: reference library file not found. Using a small dummy library instead; match results are not meaningful.')
matches_csv = OUTPUTS_DIR / 'indicator_matches.csv'
if matches_df.empty and matches_csv.exists() and len(pd.read_csv(matches_csv)) > 0:
    print(f'No indicators to match in this run; keeping the existing {matches_csv} untouched.')
else:
    matches_df.to_csv(matches_csv, index=False)
    print('Saved', matches_csv, 'with', len(matches_df), 'match records')

if not matches_df.empty:
    if EMBEDDING_MODEL and (matches_df['embedding_score'] == 0).all():
        print('WARNING: all embedding scores are 0. The embedding step failed and matching fell back to lexical only. Check that fastembed is installed (pip install fastembed) and the model can be downloaded.')
    print('\nMatch status vs ZOA reference library:')
    print(matches_df['match_status'].value_counts().to_string())
    covered = matches_df['match_status'].isin(['exact', 'close', 'related']).sum()
    print(f'\nCovered by library (exact/close/related): {covered} of {len(matches_df)}; novel candidates for library expansion: {len(matches_df) - covered}')
matches_df.head()


/Users/hiddemakimei/Desktop/git/zoa/MEAL_Proposal/version_2.1/proposal_investigated/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embeddings: using fastembed backend with model all-MiniLM-L6-v2
Saved outputs/indicator_matches.csv with 988 match records

Match status vs ZOA reference library:
match_status
novel      824
related    124
close       25
exact       15

Covered by library (exact/close/related): 164 of 988; novel candidates for library expansion: 824


,doc_id,indicator_text,level,baseline,target,means_of_verification,sector,disaggregation,unit_or_type,source_confidence,reference_id_lexical,reference_text_lexical,lexical_score,reference_id_embedding,reference_text_embedding,embedding_score,match_status,dummy_library
0,(ENGLISH version) 2024 Project proposal SelfRi...,Direct project participants CBS Kalehe,unclear,NaN,72,NaN,NaN,NaN,NaN,low,5659-well-being-questionnaire-srq-20,% of project participants that experience an i...,48.333333,5612-local-involvement-in-project-design,% of proposals where local partners were invol...,0.435775,novel,False
1,(ENGLISH version) 2024 Project proposal SelfRi...,Direct project participants VSLA Kalehe,unclear,NaN,200,NaN,NaN,NaN,NaN,low,5659-well-being-questionnaire-srq-20,% of project participants that experience an i...,47.933884,5614-use-of-a-gender-inclusion-or-power-analysis,"% of projects that utilize a gender, inclusion...",0.478790,novel,False
2,(ENGLISH version) 2024 Project proposal SelfRi...,Direct project participants PIP Idjwi,unclear,NaN,2100,NaN,NaN,NaN,NaN,low,5659-well-being-questionnaire-srq-20,% of project participants that experience an i...,47.058824,5529-awareness-of-how-to-participate-in-activi...,% of target group members aware of how they ca...,0.401829,novel,False
3,(ENGLISH version) 2024 Project proposal SelfRi...,Direct project participants G/PM Idjwi,unclear,NaN,20,NaN,NaN,NaN,gender champions,low,5368-womens-participation-in-community-groups,% of women participating in community groups,46.341463,5529-awareness-of-how-to-participate-in-activi...,% of target group members aware of how they ca...,0.507158,novel,False
4,(ENGLISH version) 2024 Project proposal SelfRi...,Direct project participants VLSA Idjwi,unclear,NaN,1100,NaN,NaN,100 1e generation and 1000 2e and 3e generation,NaN,low,5659-well-being-questionnaire-srq-20,% of project participants that experience an i...,46.666667,5529-awareness-of-how-to-participate-in-activi...,% of target group members aware of how they ca...,0.440931,novel,False


## Statistics and visualizations
Generate summary charts for indicator counts, metadata completeness, and match status.


In [11]:
def save_fig(fig, name):
    path = FIGURES_DIR / name
    fig.savefig(path, bbox_inches='tight')
    return path

if not indicators_df.empty:
    counts = indicators_df['doc_id'].value_counts().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(10, 6))
    counts.plot(kind='bar', ax=ax)
    ax.set_title('Indicators per proposal')
    ax.set_ylabel('Count')
    ax.set_xlabel('Proposal')
    save_fig(fig, 'indicators_per_proposal.png')
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 5))
    indicators_df['level'].fillna('unclear').value_counts().plot(kind='bar', ax=ax)
    ax.set_title('Indicator distribution by level')
    ax.set_ylabel('Count')
    save_fig(fig, 'indicator_level_distribution.png')
    plt.close(fig)

    sector_plot = indicators_df['sector'].fillna('unknown').value_counts()
    fig, ax = plt.subplots(figsize=(10, 5))
    sector_plot.plot(kind='bar', ax=ax)
    ax.set_title('Indicator distribution by sector')
    ax.set_ylabel('Count')
    save_fig(fig, 'indicator_sector_distribution.png')
    plt.close(fig)

    completeness = pd.DataFrame({
        'baseline': indicators_df['baseline'].notna() & (indicators_df['baseline'].astype(str) != ''),
        'target': indicators_df['target'].notna() & (indicators_df['target'].astype(str) != ''),
        'means_of_verification': indicators_df['means_of_verification'].notna() & (indicators_df['means_of_verification'].astype(str) != ''),
    }).mean().sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(8, 4))
    completeness.plot(kind='barh', ax=ax)
    ax.set_title('Metadata completeness share')
    save_fig(fig, 'metadata_completeness.png')
    plt.close(fig)

    if not matches_df.empty and 'match_status' in matches_df.columns:
        fig, ax = plt.subplots(figsize=(6, 6))
        matches_df['match_status'].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=ax)
        ax.set_ylabel('')
        ax.set_title('Match status distribution vs reference library')
        save_fig(fig, 'match_status_distribution.png')
        plt.close(fig)

    fields = ['level', 'baseline', 'target', 'means_of_verification', 'sector', 'disaggregation', 'unit_or_type']
    heatmap_data = indicators_df[fields].notna().astype(int)
    if not heatmap_data.empty:
        fig, ax = plt.subplots(figsize=(10, max(4, len(heatmap_data) * 0.15)))
        ax.imshow(heatmap_data.values.T, aspect='auto', cmap='Greens')
        ax.set_yticks(range(len(fields)))
        ax.set_yticklabels(fields)
        ax.set_xticks(range(min(len(heatmap_data), 20)))
        ax.set_xticklabels(heatmap_data.index[:20].tolist(), rotation=90)
        ax.set_title('Indicator metadata completeness heatmap (first 20 rows)')
        save_fig(fig, 'completeness_heatmap.png')
        plt.close(fig)
else:
    print('No extracted indicators available to visualize yet.')


## Phase 4: Link indicators to the ZOA reference library
Classify every extracted proposal indicator as **Present**, **Different** or **Missing** relative to the ZOA IndiKit library.

Two steps per indicator:
1. **Shortlist** the `LINK_TOP_K` most similar ZOA indicators from the whole library (303 rows) using sentence embeddings, so the LLM never has to scan the full list and every ZOA indicator has a chance to be matched.
2. **LLM judgement** on that shortlist only: *Present* means a ZOA indicator measures the same thing (differences only in population, place or numbers); *Different* means the same topic is covered but measured differently, so the proposal indicator would need adapting; *Missing* means no candidate covers the topic.

LLM responses are cached under `outputs/api_cache/link_*.json`, so re-runs are free unless `FORCE_REFRESH` is set. The returned reference id is validated against the shortlist, so the model cannot invent a ZOA indicator.


In [12]:
# Load the ZOA reference library built earlier in the notebook
from extraction_utils import load_reference_library, shortlist_reference_candidates, link_indicator_to_reference

zoa_reference_df, zoa_is_dummy = load_reference_library(str(REFERENCE_LIBRARY_PATH))
if zoa_is_dummy:
    print('WARNING: reference_library.csv not found; run the "Build the reference library" cell first. Linking against a dummy library is not meaningful.')
print(f'Loaded {len(zoa_reference_df)} ZOA reference indicators across {zoa_reference_df["sector"].nunique()} sectors')
zoa_reference_df[['indicator_id', 'indicator_text', 'sector']].head()


Loaded 303 ZOA reference indicators across 30 sectors


,indicator_id,indicator_text,sector
0,5083-individual-dietary-diversity-score,the average number of different food groups co...,Food Security
1,5089-household-dietary-diversity-score,the average number of different food groups co...,Food Security
2,5090-fsl-food-consumption-score,% of the target population with acceptable Foo...,Sector specific results of CVA
3,5091-fsl-coping-strategy-index,% of households in the highest Coping Strategy...,Sector specific results of CVA
4,5092-months-of-adequate-household-food-provisi...,the average duration of a period in the past 1...,Food Security


In [13]:
# Step 1: shortlist the closest ZOA candidates for every proposal indicator (embeddings over the whole library)
if indicators_df.empty:
    candidate_lists = []
    print('No extracted indicators to shortlist (LLM extraction may not have run).')
else:
    candidate_lists = shortlist_reference_candidates(
        indicators_df['indicator_text'].astype(str).tolist(),
        zoa_reference_df,
        k=LINK_TOP_K,
        embedding_model=EMBEDDING_MODEL,
    )
    methods = pd.Series([c[0]['method'] for c in candidate_lists if c]).value_counts().to_dict()
    print(f'Shortlisted top {LINK_TOP_K} candidates for {len(candidate_lists)} indicators (method: {methods})')
    # Preview one shortlist
    preview = candidate_lists[0]
    print(f'\nExample: "{indicators_df.iloc[0]["indicator_text"]}"')
    for c in preview:
        print(f"  {c['score']:.2f}  {c['indicator_id']}  |  {c['indicator_text'][:90]}")


Shortlisted top 5 candidates for 988 indicators (method: {'embedding': 988})

Example: "Direct project participants CBS Kalehe"
  0.44  5612-local-involvement-in-project-design  |  % of proposals where local partners were involved in the design phase.
  0.40  5611-peace-nexus-in-evaluations  |  % of project evaluations that include a question related to Peace Nexus.
  0.39  5614-use-of-a-gender-inclusion-or-power-analysis  |  % of projects that utilize a gender, inclusion or power analysis.
  0.39  5529-awareness-of-how-to-participate-in-activities  |  % of target group members aware of how they can participate in relevant project activities
  0.38  5609-projects-with-a-peace-conflict-indicator  |  % of current projects that include a peace/conflict related indicator.


In [14]:
# Step 2: LLM judgement (Present / Different / Missing) on each shortlist
if indicators_df.empty:
    print('No indicators to link.')
elif not LLM_AVAILABLE:
    print(f'LLM not available, skipping linking. {llm_status}')
else:
    print(f'Linking {len(indicators_df)} proposal indicators to the ZOA reference with {MODEL_NAME}...')
    indicators_df = indicators_df.reset_index(drop=True)
    linking_results = []
    n_cached = 0
    for i, (text, candidates) in enumerate(zip(indicators_df['indicator_text'].astype(str), candidate_lists)):
        result = link_indicator_to_reference(text, candidates, model_name=MODEL_NAME)
        n_cached += int(result.get('cached', False))
        linking_results.append({
            'zoa_match_id': result['reference_id'],
            'zoa_match': result['reference_text'],
            'match_status': result['status'],
            'confidence': result['confidence'],
            'reason': result['reason'],
            'shortlist_top_score': candidates[0]['score'] if candidates else None,
        })
        if (i + 1) % 100 == 0:
            print(f'  Linked {i + 1} / {len(indicators_df)} (cached so far: {n_cached})')

    linking_df = pd.DataFrame(linking_results)
    # Drop any earlier linking columns so this cell is safe to re-run
    indicators_df = indicators_df.drop(columns=[c for c in linking_df.columns if c in indicators_df.columns])
    indicators_df = pd.concat([indicators_df, linking_df], axis=1)

    n_err = (indicators_df['match_status'] == 'Error').sum()
    print(f'Linked {len(indicators_df)} indicators ({n_cached} from cache). Errors: {n_err}')
    if n_err:
        print('First error reasons:')
        print(indicators_df.loc[indicators_df['match_status'] == 'Error', 'reason'].head(3).to_string(index=False))


Linking 988 proposal indicators to the ZOA reference with gpt-4o-mini...
  Linked 100 / 988 (cached so far: 0)
  Linked 200 / 988 (cached so far: 0)
  Linked 300 / 988 (cached so far: 1)
  Linked 400 / 988 (cached so far: 9)
  Linked 500 / 988 (cached so far: 10)
  Linked 600 / 988 (cached so far: 12)
  Linked 700 / 988 (cached so far: 12)
  Linked 800 / 988 (cached so far: 15)
  Linked 900 / 988 (cached so far: 29)
Linked 988 indicators (35 from cache). Errors: 20
First error reasons:
APIConnectionError: Connection error.
APIConnectionError: Connection error.
APIConnectionError: Connection error.


In [15]:
# Save indicators with linking results
indicators_csv_linked = OUTPUTS_DIR / 'indicators_extracted_linked.csv'
if 'match_status' not in indicators_df.columns and indicators_csv_linked.exists() and len(pd.read_csv(indicators_csv_linked)) > 0:
    print(f'Linking did not run in this session; keeping the existing {indicators_csv_linked} untouched.')
else:
    indicators_df.to_csv(indicators_csv_linked, index=False)
    print(f'Saved: {indicators_csv_linked}')

if not indicators_df.empty and 'match_status' in indicators_df.columns:
    print('\n=== LINKING SUMMARY ===')
    print(indicators_df['match_status'].value_counts().to_string())
    print(f"\nAverage confidence: {indicators_df['confidence'].mean():.2f}")
    display(indicators_df[['doc_id', 'indicator_text', 'match_status', 'zoa_match', 'confidence', 'reason']].head(10))


Saved: outputs/indicators_extracted_linked.csv

=== LINKING SUMMARY ===
match_status
Missing      606
Present      204
Different    158
Error         20

Average confidence: 0.85


,doc_id,indicator_text,match_status,zoa_match,confidence,reason
0,(ENGLISH version) 2024 Project proposal SelfRi...,Direct project participants CBS Kalehe,Missing,NaN,0.9,None of the reference indicators specifically ...
1,(ENGLISH version) 2024 Project proposal SelfRi...,Direct project participants VSLA Kalehe,Missing,NaN,0.9,None of the reference indicators specifically ...
2,(ENGLISH version) 2024 Project proposal SelfRi...,Direct project participants PIP Idjwi,Missing,NaN,0.9,The proposal indicator specifically measures d...
3,(ENGLISH version) 2024 Project proposal SelfRi...,Direct project participants G/PM Idjwi,Missing,NaN,0.9,None of the reference indicators specifically ...
4,(ENGLISH version) 2024 Project proposal SelfRi...,Direct project participants VLSA Idjwi,Missing,NaN,0.9,None of the reference indicators specifically ...
5,2.1 Annex I_Description of the Action_451054_Z...,Total number of persons targeted,Missing,NaN,0.9,None of the reference indicators specifically ...
6,2.1 Annex I_Description of the Action_451054_Z...,Total number of females targeted,Missing,NaN,0.9,None of the reference indicators specifically ...
7,2.1 Annex I_Description of the Action_451054_Z...,Total number of IDPs targeted,Missing,NaN,0.9,None of the reference indicators specifically ...
8,2.1 Annex I_Description of the Action_451054_Z...,Total number of PWDs targeted,Missing,NaN,0.9,None of the reference indicators specifically ...
9,2.1 Annex I_Description of the Action_451054_Z...,Total number of immigrants targeted,Missing,NaN,0.9,None of the reference indicators specifically ...


## Manual check: detected indicators per proposal
Review table for checking the extraction and linking by hand. Each row is one detected indicator with the proposal it came from, what the LLM extracted (level, baseline, target) and, when Phase 4 has run, the ZOA match and status.

Set `REVIEW_PROPOSAL` to a proposal file name (or part of it) to show only that proposal, or leave it `None` to list all. The full table is also written to `outputs/manual_check.xlsx` with one sheet per proposal and empty `reviewer_verdict` / `reviewer_note` columns to fill in.

In [16]:
REVIEW_PROPOSAL = None   # e.g. 'Syria Joint Response' or '2023_00217'; None shows every proposal

review_cols = ['doc_id', 'indicator_text', 'level', 'baseline', 'target', 'means_of_verification', 'sector',
               'zoa_match', 'match_status', 'confidence', 'reason']
review_cols = [c for c in review_cols if c in indicators_df.columns]
review_df = (indicators_df[review_cols]
             .rename(columns={'doc_id': 'proposal'})
             .sort_values(['proposal', 'indicator_text'])
             .reset_index(drop=True))
review_df.insert(0, 'n', review_df.groupby('proposal').cumcount() + 1)
review_df['reviewer_verdict'] = ''   # e.g. correct / not an indicator / wrong match
review_df['reviewer_note'] = ''

# Overview: how many indicators were detected in each proposal
per_proposal = review_df.groupby('proposal').size().rename('indicators_detected').sort_values(ascending=False)
print(f'{len(review_df)} indicators detected across {per_proposal.shape[0]} proposals')
display(per_proposal.to_frame())

# Save the review workbook (one sheet per proposal + a combined sheet)
manual_check_xlsx = OUTPUTS_DIR / 'manual_check.xlsx'
with pd.ExcelWriter(manual_check_xlsx, engine='openpyxl') as writer:
    review_df.to_excel(writer, sheet_name='All proposals', index=False)
    used_names = set()
    for proposal, group in review_df.groupby('proposal'):
        name = ''.join(ch if ch not in '[]:*?/\\' else '_' for ch in str(proposal))[:28]
        base, k = name, 1
        while name in used_names:
            k += 1
            name = f'{base[:25]}_{k}'
        used_names.add(name)
        group.to_excel(writer, sheet_name=name or 'proposal', index=False)
print(f'Saved review workbook: {manual_check_xlsx}')

# Show the indicators for one proposal (or all) with full text visible
shown = review_df if not REVIEW_PROPOSAL else review_df[review_df['proposal'].str.contains(REVIEW_PROPOSAL, case=False, regex=False)]
if REVIEW_PROPOSAL and shown.empty:
    print(f'No proposal name contains "{REVIEW_PROPOSAL}". Available names:')
    print('\n'.join(per_proposal.index))
with pd.option_context('display.max_rows', 500, 'display.max_colwidth', 120, 'display.width', 250):
    display(shown.drop(columns=['reviewer_verdict', 'reviewer_note']))


988 indicators detected across 75 proposals


,indicators_detected
proposal,
Final Full Proposal.pdf,93
290903_A-GRIP_improved_fullproposal excerpts for CD partners.pdf,51
2023_00217_MR_01_01_18-Jan-2024.pdf,47
ARFSA-Annex-I-Final Narrative Proposal.pdf,42
"ZOA Revised Proposal Tigray Emergency Response 2021-07-27, BHA.pdf",42
...,...
Emergency Shelter Assistance Concept Note June 2024 final.pdf,2
Khartoum Responses_FoCo Proposal_Revised 2026.docx,2
SSD2405 - Project proposal Pharus - Resilience in South Sudan Y5.docx,2


Saved review workbook: outputs/manual_check.xlsx


,n,proposal,indicator_text,level,baseline,target,means_of_verification,sector,zoa_match,match_status,confidence,reason
0,1,(ENGLISH version) 2024 Project proposal SelfRise - Congo - NL ZBA.pdf,Direct project participants CBS Kalehe,unclear,NaN,72,NaN,NaN,NaN,Missing,0.9,None of the reference indicators specifically measure the number of direct project participants.
1,2,(ENGLISH version) 2024 Project proposal SelfRise - Congo - NL ZBA.pdf,Direct project participants G/PM Idjwi,unclear,NaN,20,NaN,NaN,NaN,Missing,0.9,None of the reference indicators specifically measure the number of direct project participants.
2,3,(ENGLISH version) 2024 Project proposal SelfRise - Congo - NL ZBA.pdf,Direct project participants PIP Idjwi,unclear,NaN,2100,NaN,NaN,NaN,Missing,0.9,"The proposal indicator specifically measures direct project participants, which is not addressed by any of the refer..."
3,4,(ENGLISH version) 2024 Project proposal SelfRise - Congo - NL ZBA.pdf,Direct project participants VLSA Idjwi,unclear,NaN,1100,NaN,NaN,NaN,Missing,0.9,None of the reference indicators specifically measure the number of direct project participants in VLSA Idjwi.
4,5,(ENGLISH version) 2024 Project proposal SelfRise - Congo - NL ZBA.pdf,Direct project participants VSLA Kalehe,unclear,NaN,200,NaN,NaN,NaN,Missing,0.9,None of the reference indicators specifically measure the number of direct project participants in a VSLA context.
...,...,...,...,...,...,...,...,...,...,...,...,...
983,25,ZOA-we-are-able-full-proposal-v6.pdf,"The percentage of surveyed women and girls say they have improved social inclusion in their communities, measured by...",unclear,NaN,NaN,NaN,NaN,% of planned measures for mainstreaming social inclusion that were implemented,Different,0.7,"The proposal indicator focuses on perceived social inclusion among women and girls, while the closest reference indi..."
984,26,ZOA-we-are-able-full-proposal-v6.pdf,The percentage of surveyed women and girls say they have improved social inclusion in their communities.,unclear,NaN,70%,"Tracer information for members of the grass root groups, to be measured by the Participation Scale Questionnaire.",NaN,% of planned measures for mainstreaming social inclusion that were implemented,Different,0.7,"While both indicators relate to social inclusion and community cohesion, the proposal indicator specifically measure..."
985,27,ZOA-we-are-able-full-proposal-v6.pdf,[Are the following functional responsibilities performed by different units or persons: (a) ordering; (b) payment; (...,unclear,There is one person responsible for all described duties.,Segregation of described duties is formalized and accordingly implemented by sufficient people.,NaN,NaN,NaN,Missing,0.9,"None of the reference indicators address the functional responsibilities related to ordering, payment, recording tra..."
986,28,ZOA-we-are-able-full-proposal-v6.pdf,"[Describe the basis of accounting (e.g. cash, accrual)?]",unclear,NaN,NaN,NaN,NaN,NaN,Missing,0.9,"The proposal indicator regarding the basis of accounting does not align with any of the reference indicators, which ..."


## Phase 5: Generate Summary Tables
Create summary statistics by proposal and sector.

In [17]:
# Summary by proposal
if 'match_status' in indicators_df.columns:
    proposal_summary = indicators_df.groupby('doc_id').apply(
        lambda group: pd.Series({
            'Total Indicators': len(group),
            'Present': (group['match_status'] == 'Present').sum(),
            'Different': (group['match_status'] == 'Different').sum(),
            'Missing': (group['match_status'] == 'Missing').sum(),
        })
    ).reset_index()
    
    # Add percentages
    proposal_summary['% Present'] = (proposal_summary['Present'] / proposal_summary['Total Indicators'] * 100).round(1)
    proposal_summary['% Different'] = (proposal_summary['Different'] / proposal_summary['Total Indicators'] * 100).round(1)
    proposal_summary['% Missing'] = (proposal_summary['Missing'] / proposal_summary['Total Indicators'] * 100).round(1)
    
    # Save
    proposal_summary_csv = OUTPUTS_DIR / 'proposal_indicator_summary.csv'
    proposal_summary.to_csv(proposal_summary_csv, index=False)
    print(f'Saved: {proposal_summary_csv}\n')
    print(proposal_summary)

Saved: outputs/proposal_indicator_summary.csv

                                               doc_id  Total Indicators  \
0   (ENGLISH version) 2024 Project proposal SelfRi...                 5   
1   2.1 Annex I_Description of the Action_451054_Z...                22   
2     2022_03_21_FNS_IWRM_Revised version_ TRIDE.docx                 4   
3                 2023_00217_MR_01_01_18-Jan-2024.pdf                47   
4       231031 PCM_SJR_2024-2026_Final_Narrative.docx                13   
..                                                ...               ...   
70  ZBA_Ruyigi_RESILIENCE_Proposal (modified propo...                12   
71  ZOA - Group Cash Transfers, Taiz, Yemen - Proj...                14   
72  ZOA Revised Proposal Tigray Emergency Response...                42   
73  ZOA-230724-YJR 2024-2026 Narrative 05 Oct 2023...                 2   
74               ZOA-we-are-able-full-proposal-v6.pdf                29   

    Present  Different  Missing  % Present  % Differ

In [18]:
# Summary by sector
if 'match_status' in indicators_df.columns and 'sector' in indicators_df.columns:
    sector_summary = indicators_df.groupby('sector').apply(
        lambda group: pd.Series({
            'Total Indicators': len(group),
            'Present': (group['match_status'] == 'Present').sum(),
            'Different': (group['match_status'] == 'Different').sum(),
            'Missing': (group['match_status'] == 'Missing').sum(),
        })
    ).reset_index()
    
    # Add percentages
    sector_summary['% Present'] = (sector_summary['Present'] / sector_summary['Total Indicators'] * 100).round(1)
    sector_summary['% Different'] = (sector_summary['Different'] / sector_summary['Total Indicators'] * 100).round(1)
    sector_summary['% Missing'] = (sector_summary['Missing'] / sector_summary['Total Indicators'] * 100).round(1)
    
    # Save
    sector_summary_csv = OUTPUTS_DIR / 'sector_indicator_summary.csv'
    sector_summary.to_csv(sector_summary_csv, index=False)
    print(f'Saved: {sector_summary_csv}\n')
    print(sector_summary)

Saved: outputs/sector_indicator_summary.csv

                                                sector  Total Indicators  \
0                                       Accountability                 1   
1                                          Agriculture                 3   
2                          Agriculture et Sylviculture                 1   
3    Aide alimentaire à des fins de développement/s...                 1   
4                                     Amplified voices                 1   
..                                                 ...               ...   
110                                             market                 2   
111                                          nutrition                 5   
112                                            poultry                 1   
113                                         protection                 3   
114                                         technology                 2   

     Present  Different  Missing  % Presen

In [19]:
# Generate simple insights
if 'match_status' in indicators_df.columns:
    total_indicators = len(indicators_df)
    present_count = (indicators_df['match_status'] == 'Present').sum()
    different_count = (indicators_df['match_status'] == 'Different').sum()
    missing_count = (indicators_df['match_status'] == 'Missing').sum()
    
    insights = f"""
PROPOSAL INDICATOR LINKING SUMMARY
═══════════════════════════════════

Overall Statistics:
  Total proposals analyzed: {indicators_df['doc_id'].nunique()}
  Total indicators extracted: {total_indicators}
  
Linking Results:
  Present (matched to ZOA): {present_count} ({present_count/total_indicators*100:.1f}%)
  Different (similar but adapted): {different_count} ({different_count/total_indicators*100:.1f}%)
  Missing (no ZOA equivalent): {missing_count} ({missing_count/total_indicators*100:.1f}%)

Average Confidence: {indicators_df['confidence'].mean():.2f}

Top 10 Most Frequently Missing Indicators:
{indicators_df[indicators_df['match_status'] == 'Missing']['indicator_text'].value_counts().head(10).to_string()}

Top 10 Most Frequently Adapted Indicators:
{indicators_df[indicators_df['match_status'] == 'Different']['zoa_match'].value_counts().head(10).to_string()}
"""
    
    # Save insights
    insights_txt = OUTPUTS_DIR / 'insights_summary.txt'
    with open(insights_txt, 'w') as f:
        f.write(insights)
    
    print(insights)
    print(f'\nSaved: {insights_txt}')


PROPOSAL INDICATOR LINKING SUMMARY
═══════════════════════════════════

Overall Statistics:
  Total proposals analyzed: 75
  Total indicators extracted: 988

Linking Results:
  Present (matched to ZOA): 204 (20.6%)
  Different (similar but adapted): 158 (16.0%)
  Missing (no ZOA equivalent): 606 (61.3%)

Average Confidence: 0.85

Top 10 Most Frequently Missing Indicators:
indicator_text
Number of people provided with resources to protect and start rebuilding livelihood assets                                                             3
Number of small-scale food producers who progressively realize a living income.                                                                        2
Number of small-scale food producers whose livelihood became more resilient to shocks.                                                                 2
Number of small-scale food producers indirectly reached.                                                                                              